In [14]:
# 1. 개인 Google Drive의 SAS 원자료 읽기
import os, sys, subprocess, importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

cols = ['INT_SPWD','INT_SPWD_TM','E_S_RCRD','SEX','GRADE','E_SES','W','STRATA','CLUSTER']
DATA_DIR = '/content/drive/MyDrive/hdac_data'
SAS_PATH = f'{DATA_DIR}/kyrbs2025.sas7bdat'

if 'google.colab' in sys.modules or Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')

if Path(SAS_PATH).is_file():
    if importlib.util.find_spec('pyreadstat') is None:
        subprocess.check_call([sys.executable,'-m','pip','-q','install','pyreadstat'])
    import pyreadstat
    raw, meta = pyreadstat.read_sas7bdat(SAS_PATH, usecols=cols)
    SOURCE = '2025 청소년건강행태조사 SAS 원자료'
else:
    # Colab에서는 개인 Drive 파일이 없으면 중단합니다. 로컬 검증용 CSV는 별도 환경변수 지정 시만 사용.
    local_csv = os.environ.get('KYRBS_LOCAL_CSV')
    if not local_csv or not Path(local_csv).is_file():
        raise FileNotFoundError(f'{SAS_PATH} 에 원자료가 없습니다. 개인 Drive hdac_data 폴더에 파일을 넣으세요.')
    raw = pd.read_csv(local_csv, usecols=cols, low_memory=False)
    SOURCE = '2025 청소년건강행태조사 원자료 CSV (SAS와 같은 변수를 사용한 검증)'

for c in ['INT_SPWD','INT_SPWD_TM','E_S_RCRD','SEX','GRADE','E_SES','W']:
    raw[c] = pd.to_numeric(raw[c], errors='coerce')
raw['STRATA'] = raw['STRATA'].astype(str)
raw['PSU_KEY'] = raw['STRATA'] + '|' + raw['CLUSTER'].astype(str)
print('자료:',SOURCE, '| 원자료:',len(raw),'명', '| 변수:',len(cols),'개')
print('층:',raw['STRATA'].nunique(),'개 | 층 내 PSU:',raw['PSU_KEY'].nunique(),'개')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
자료: 2025 청소년건강행태조사 SAS 원자료 | 원자료: 54170 명 | 변수: 9 개
층: 106 개 | 층 내 PSU: 796 개


In [8]:
# 2. 10주차와 동일한 분석대상 / 노출 / 결과 / 보정변수 설정
raw['phone_min'] = np.where(raw['INT_SPWD'].eq(1), 0.0, raw['INT_SPWD_TM'])
raw['eligible'] = raw['GRADE'].isin(range(1,7))
raw['exposure_ok'] = raw['phone_min'].ge(0) & np.isfinite(raw['phone_min'])
raw['outcome_ok'] = raw['E_S_RCRD'].isin([1,2,3,4,5])
raw['in_analysis'] = raw['eligible'] & raw['exposure_ok'] & raw['outcome_ok']
raw['complete_case'] = raw['in_analysis'] & raw[['SEX','GRADE','E_SES']].notna().all(axis=1) & raw['SEX'].isin([1,2]) & raw['GRADE'].isin(range(1,7)) & raw['E_SES'].isin([1,2,3,4,5]) & raw['W'].gt(0)
raw['high_grade'] = (raw['E_S_RCRD'].isin([1,2])).astype(int)
raw['phone_hr'] = raw['phone_min']/60.0
# right=False에서 0을 별도 구분하기 위한 직접 할당
raw['phone_group'] = pd.Series(index=raw.index,dtype='object')
raw.loc[raw['phone_min'].eq(0),'phone_group']='미사용'
raw.loc[raw['phone_min'].gt(0)&raw['phone_min'].lt(120),'phone_group']='0분 초과~2시간 미만'
raw.loc[raw['phone_min'].ge(120)&raw['phone_min'].lt(240),'phone_group']='2~4시간 미만'
raw.loc[raw['phone_min'].ge(240)&raw['phone_min'].lt(360),'phone_group']='4~6시간 미만'
raw.loc[raw['phone_min'].ge(360),'phone_group']='6시간 이상'
GROUP_ORDER=['미사용','0분 초과~2시간 미만','2~4시간 미만','4~6시간 미만','6시간 이상']

n_total = len(raw)
n_table1 = int(raw['in_analysis'].sum())
n_cc = int(raw['complete_case'].sum())
n_miss = n_table1-n_cc
print(f'전체 n={n_total:,}, Table 1 n={n_table1:,}, 완전 사례 n={n_cc:,}, 보정변수 결측으로 제외 n={n_miss:,}')
print('완전 사례의 노출 집단별 인원:')
display(raw.loc[raw.complete_case,'phone_group'].value_counts().reindex(GROUP_ORDER).rename('n').to_frame())
assert n_table1 == 54164, '10주차 Table 1 인원과 다릅니다. 원자료·변수 코딩을 점검하세요.'
assert n_cc == 54158, '예상 완전 사례 54,158명과 다릅니다. 결측 코딩을 점검하세요.'
assert raw.loc[raw.complete_case, 'phone_group'].notna().all()

전체 n=54,170, Table 1 n=54,164, 완전 사례 n=54,158, 보정변수 결측으로 제외 n=6
완전 사례의 노출 집단별 인원:


,n
phone_group,
미사용,1688
0분 초과~2시간 미만,4865
2~4시간 미만,18203
4~6시간 미만,16167
6시간 이상,13235


In [9]:
# 3. 복합표본설계 기반 가중 로지스틱 회귀 함수
# 점추정: 조사 가중치가 적용된 로지스틱 회귀
# SE: 전 원자료의 층/PSU 구조를 보존하고, 분석 도메인 외 대상자 score=0으로 처리한
#     층화 PSU 기반 Taylor 선형화 (with-replacement), 자유도=PSU수-층수
import statsmodels.api as sm
from scipy.stats import t as tdist

psu_info = raw[['STRATA','PSU_KEY']].drop_duplicates().reset_index(drop=True)
psu_info['psu_idx'] = np.arange(len(psu_info))
psu_codes = raw['PSU_KEY'].map(psu_info.set_index('PSU_KEY')['psu_idx']).to_numpy(dtype=int)
strata_groups = [g['psu_idx'].to_numpy(dtype=int) for _,g in psu_info.groupby('STRATA',sort=False)]
if any(len(ix)<2 for ix in strata_groups):
    raise ValueError('단독 PSU 층 발생: 분산추정 규칙 별도 검토 필요')
design_df = len(psu_info) - len(strata_groups)
print('설계 자유도:', design_df)

def fit_svy_logit(xdf, term_names):
    mask=raw['complete_case'].to_numpy()
    Xfull = sm.add_constant(xdf.astype(float), has_constant='add')
    X = Xfull.loc[mask].to_numpy(dtype=float)
    y = raw.loc[mask,'high_grade'].to_numpy(dtype=float)
    w0 = raw.loc[mask,'W'].to_numpy(dtype=float)
    w = w0/w0.mean()  # 계수/샌드위치 분산은 상수 배율에 불변
    res=sm.GLM(y,X,family=sm.families.Binomial(),freq_weights=w).fit(maxiter=150)
    pred=np.asarray(res.predict(X))
    H=(X.T*(w*pred*(1-pred)))@X
    Hinv=np.linalg.inv(H)
    # 전체 PSU 배열에 도메인 외 점수 0을 부여. 인원에 따라 자동으로 삭제하지 않음.
    score_all=np.zeros((len(raw),X.shape[1]),dtype=float)
    score_all[mask,:]=(w*(y-pred))[:,None]*X
    psu_scores=np.zeros((len(psu_info),X.shape[1]))
    np.add.at(psu_scores,psu_codes,score_all)
    meat=np.zeros((X.shape[1],X.shape[1]),dtype=float)
    for ix in strata_groups:
        u=psu_scores[ix,:]
        centered=u-u.mean(axis=0)
        meat+=(len(ix)/(len(ix)-1))*(centered.T@centered)
    cov=Hinv@meat@Hinv
    se=np.sqrt(np.diag(cov))
    b=np.asarray(res.params)
    critical=tdist.ppf(.975,design_df)
    pvals=2*tdist.sf(abs(b/se),design_df)
    out=pd.DataFrame({'변수':['절편']+list(term_names),'OR':np.exp(b),'CI_low':np.exp(b-critical*se),
                      'CI_high':np.exp(b+critical*se),'p':pvals,'beta':b,'SE_beta':se})
    return out

# 2개의 주모형은 항상 같은 54,158명 도메인으로 적합
X_unadj = raw[['phone_hr']]
X_adj = pd.DataFrame({'phone_hr':raw['phone_hr']},index=raw.index)
for col,valid in [('SEX',[1,2]),('GRADE',[1,2,3,4,5,6]),('E_SES',[1,2,3,4,5])]:
    for level in valid[1:]:
        X_adj[f'{col}_{level}'] = raw[col].eq(level).astype(float)
result1=fit_svy_logit(X_unadj,['phone_hr'])
result2=fit_svy_logit(X_adj,list(X_adj.columns))
print('모형 1, 2 추정 완료')

설계 자유도: 690
모형 1, 2 추정 완료


In [10]:
# 4. Table 2: 1시간 증가당 OR, 95% CI, p값

def pretty(est):
    r=est.loc[est['변수']=='phone_hr'].iloc[0]
    return f"{r.OR:.3f} ({r.CI_low:.3f}–{r.CI_high:.3f})",f"{r.p:.4f}" if r.p>=.0001 else '<0.0001'

m1,p1=pretty(result1)
m2,p2=pretty(result2)
main_table=pd.DataFrame([
    ['스마트폰 사용시간 1시간 증가',m1,p1,m2,p2]
],columns=['노출','모형 1 OR (95% CI)','모형 1 p값','모형 2 OR (95% CI)','모형 2 p값'])
display(main_table)
print('두 모형 공통 도메인 n:',n_cc)
print('참고: 학업성적 상·중상=1, 그 외 0. OR>1은 상·중상의 오즈 증가.')

,노출,모형 1 OR (95% CI),모형 1 p값,모형 2 OR (95% CI),모형 2 p값
0,스마트폰 사용시간 1시간 증가,0.830 (0.821–0.839),<0.0001,0.854 (0.845–0.864),<0.0001


두 모형 공통 도메인 n: 54158
참고: 학업성적 상·중상=1, 그 외 0. OR>1은 상·중상의 오즈 증가.


In [11]:
# 5. 보조분석: Table 1의 5개 노출 범주 (미사용 기준)
Xcat=pd.DataFrame(index=raw.index)
for group in GROUP_ORDER[1:]:
    Xcat[group]=raw['phone_group'].eq(group).astype(float)
cat1=fit_svy_logit(Xcat,list(Xcat.columns))
Xcat_adj=pd.concat([Xcat,X_adj.drop(columns='phone_hr')],axis=1)
cat2=fit_svy_logit(Xcat_adj,list(Xcat_adj.columns))
rows=[['미사용 (기준)','1.00 (기준)','—','1.00 (기준)','—']]
for group in GROUP_ORDER[1:]:
    a=cat1.loc[cat1['변수']==group].iloc[0]; b=cat2.loc[cat2['변수']==group].iloc[0]
    pf=lambda v: f'{v:.4f}' if v>=.0001 else '<0.0001'
    rows.append([group,f'{a.OR:.3f} ({a.CI_low:.3f}–{a.CI_high:.3f})',pf(a.p),
                       f'{b.OR:.3f} ({b.CI_low:.3f}–{b.CI_high:.3f})',pf(b.p)])
cat_table=pd.DataFrame(rows,columns=main_table.columns)
display(cat_table)

,노출,모형 1 OR (95% CI),모형 1 p값,모형 2 OR (95% CI),모형 2 p값
0,미사용 (기준),1.00 (기준),—,1.00 (기준),—
1,0분 초과~2시간 미만,1.761 (1.547–2.005),<0.0001,1.473 (1.274–1.705),<0.0001
2,2~4시간 미만,0.982 (0.875–1.102),0.7585,0.955 (0.835–1.091),0.4960
3,4~6시간 미만,0.583 (0.519–0.656),<0.0001,0.618 (0.539–0.708),<0.0001
4,6시간 이상,0.357 (0.316–0.403),<0.0001,0.396 (0.345–0.455),<0.0001


In [12]:
# 6. 결과 해석 기초 비교 + 보고서 각주 생성
b1=result1.loc[result1['변수']=='phone_hr'].iloc[0]
b2=result2.loc[result2['변수']=='phone_hr'].iloc[0]
change_pct=(b2.OR/b1.OR-1)*100
print(f'1시간당 OR: 무보정 {b1.OR:.3f} → 보정 {b2.OR:.3f} (OR 상대변화 {change_pct:+.1f}%)')
print('주 분석: 학업성적 상·중상 vs 중·중하·하; 1시간 증가당 OR.')
print('각주: W 가중치, STRATA 층화, CLUSTER 집락; Taylor 선형화, 층 내 PSU 중심화, FPC 미적용.')
print('각주: 도메인 유지, 경제상태 결측 6명 때문에 두 모형 모두 동일한 완전 사례 54,158명.')
print('각주: 모형1 노출만; 모형2 노출+성별(기준 1)+학년(기준 1)+경제상태(기준 1).')
print('각주: 설계 자유도=',design_df,'; 양측 t검정과 95% CI.')

1시간당 OR: 무보정 0.830 → 보정 0.854 (OR 상대변화 +2.9%)
주 분석: 학업성적 상·중상 vs 중·중하·하; 1시간 증가당 OR.
각주: W 가중치, STRATA 층화, CLUSTER 집락; Taylor 선형화, 층 내 PSU 중심화, FPC 미적용.
각주: 도메인 유지, 경제상태 결측 6명 때문에 두 모형 모두 동일한 완전 사례 54,158명.
각주: 모형1 노출만; 모형2 노출+성별(기준 1)+학년(기준 1)+경제상태(기준 1).
각주: 설계 자유도= 690 ; 양측 t검정과 95% CI.


In [13]:
# 7. 제출용 Markdown과 CSV 저장: Colab /content/11week_outputs/
from pathlib import Path
OUT = Path('/content/11week_outputs') if Path('/content').exists() else Path('/mnt/data/11week_outputs')
OUT.mkdir(parents=True, exist_ok=True)
main_table.to_csv(OUT/'11_table2_main.csv',index=False,encoding='utf-8-sig')
cat_table.to_csv(OUT/'11_table2_categories.csv',index=False,encoding='utf-8-sig')

def markdown_table(df):
    return df.to_markdown(index=False)
report=f"""# C-2. Table 2 — 무보정·보정 회귀

## 분석 방법

제21차(2025년) 청소년건강행태조사에서 주중 스마트폰 사용시간과 주관적 학업성적 간의 관련성을 평가하였다. 결과변수는 학업성적 상·중상(1) 대 중·중하·하(0)로 이분화하였다. 모형 1은 주중 스마트폰 사용시간만 포함하고, 모형 2는 C-1 DAG에 근거하여 성별·학년·경제상태를 보정하였다.

## Table 2. 스마트폰 사용시간 1시간 증가당 높은 학업성적의 오즈비

{markdown_table(main_table)}

## 보조분석. Table 1의 스마트폰 사용시간 5개 집단

{markdown_table(cat_table)}

## 분석 인원 및 각주

- 10주차 Table 1 대상자: **{n_table1:,}명**; 11주차 완전 사례: **{n_cc:,}명** (경제상태 결측으로 **{n_miss:,}명** 제외). 두 주모형은 동일한 사람에게 적용하였다.
- **노출변수:** 스마트폰 사용시간 1시간(60분) 증가; 보조분석의 기준은 미사용군이다.
- **결과변수:** 학업성적 '상·중상'=1, 나머지=0. OR>1은 높은 학업성적에 속할 오즈가 크다는 뜻이다.
- **설계:** 가중치 W, 층화변수 STRATA, 집락변수 CLUSTER를 반영하였다. 분산은 원자료 층/PSU를 보존하고 도메인 외 사람의 score를 0으로 둔 Taylor 선형화로 추정했다. FPC 미적용.
- **검정:** 층화 PSU 기반 설계 자유도 {design_df}, 양측 t검정 및 95% 신뢰구간.
- **모형:** 모형 1은 무보정, 모형 2는 성별·학년·경제상태 보정. 보정변수별 회귀계수는 주 결과표에 생략했다.
- **결측:** 노출·결과 결측 제외 후, 세 보정변수 중 결측이 있는 대상자는 양 모형의 공통 분석 도메인에서 제외했다. 원자료의 행 자체는 삭제하지 않았다.
- **자료:** {SOURCE}.

## 결과 요약 (C-3에서 상세 해석)

스마트폰 사용시간 1시간 증가에 대한 오즈비는 무보정 {b1.OR:.3f}, 보정 {b2.OR:.3f}였으며, 보정 후 OR은 무보정 OR에 비해 {change_pct:+.1f}% 변화하였다. 횡단면 관찰자료이므로 인과효과가 아닌 관련성으로 해석한다.
"""
(OUT/'11_C2_Table2.md').write_text(report,encoding='utf-8')
print('저장 완료:',list(map(str,OUT.iterdir())))
display(Markdown(report))

저장 완료: ['/content/11week_outputs/11_C2_Table2.md', '/content/11week_outputs/11_table2_categories.csv', '/content/11week_outputs/11_table2_main.csv']


# C-2. Table 2 — 무보정·보정 회귀

## 분석 방법

제21차(2025년) 청소년건강행태조사에서 주중 스마트폰 사용시간과 주관적 학업성적 간의 관련성을 평가하였다. 결과변수는 학업성적 상·중상(1) 대 중·중하·하(0)로 이분화하였다. 모형 1은 주중 스마트폰 사용시간만 포함하고, 모형 2는 C-1 DAG에 근거하여 성별·학년·경제상태를 보정하였다.

## Table 2. 스마트폰 사용시간 1시간 증가당 높은 학업성적의 오즈비

| 노출                         | 모형 1 OR (95% CI)   | 모형 1 p값   | 모형 2 OR (95% CI)   | 모형 2 p값   |
|:-----------------------------|:---------------------|:-------------|:---------------------|:-------------|
| 스마트폰 사용시간 1시간 증가 | 0.830 (0.821–0.839)  | <0.0001      | 0.854 (0.845–0.864)  | <0.0001      |

## 보조분석. Table 1의 스마트폰 사용시간 5개 집단

| 노출                | 모형 1 OR (95% CI)   | 모형 1 p값   | 모형 2 OR (95% CI)   | 모형 2 p값   |
|:--------------------|:---------------------|:-------------|:---------------------|:-------------|
| 미사용 (기준)       | 1.00 (기준)          | —            | 1.00 (기준)          | —            |
| 0분 초과~2시간 미만 | 1.761 (1.547–2.005)  | <0.0001      | 1.473 (1.274–1.705)  | <0.0001      |
| 2~4시간 미만        | 0.982 (0.875–1.102)  | 0.7585       | 0.955 (0.835–1.091)  | 0.4960       |
| 4~6시간 미만        | 0.583 (0.519–0.656)  | <0.0001      | 0.618 (0.539–0.708)  | <0.0001      |
| 6시간 이상          | 0.357 (0.316–0.403)  | <0.0001      | 0.396 (0.345–0.455)  | <0.0001      |

## 분석 인원 및 각주

- 10주차 Table 1 대상자: **54,164명**; 11주차 완전 사례: **54,158명** (경제상태 결측으로 **6명** 제외). 두 주모형은 동일한 사람에게 적용하였다.
- **노출변수:** 스마트폰 사용시간 1시간(60분) 증가; 보조분석의 기준은 미사용군이다.
- **결과변수:** 학업성적 '상·중상'=1, 나머지=0. OR>1은 높은 학업성적에 속할 오즈가 크다는 뜻이다.
- **설계:** 가중치 W, 층화변수 STRATA, 집락변수 CLUSTER를 반영하였다. 분산은 원자료 층/PSU를 보존하고 도메인 외 사람의 score를 0으로 둔 Taylor 선형화로 추정했다. FPC 미적용.
- **검정:** 층화 PSU 기반 설계 자유도 690, 양측 t검정 및 95% 신뢰구간.
- **모형:** 모형 1은 무보정, 모형 2는 성별·학년·경제상태 보정. 보정변수별 회귀계수는 주 결과표에 생략했다.
- **결측:** 노출·결과 결측 제외 후, 세 보정변수 중 결측이 있는 대상자는 양 모형의 공통 분석 도메인에서 제외했다. 원자료의 행 자체는 삭제하지 않았다.
- **자료:** 2025 청소년건강행태조사 SAS 원자료.

## 결과 요약 (C-3에서 상세 해석)

스마트폰 사용시간 1시간 증가에 대한 오즈비는 무보정 0.830, 보정 0.854였으며, 보정 후 OR은 무보정 OR에 비해 +2.9% 변화하였다. 횡단면 관찰자료이므로 인과효과가 아닌 관련성으로 해석한다.
